<a href="https://colab.research.google.com/github/jRicardo2003/etl-data-pipeline2507232022/blob/main/notebooks/Actvidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/jRicardo2003/etl-data-pipeline2507232022/refs/heads/main/data/raw/G_actividad.csv"

df = pd.read_csv(url)

df.head()

,id_actividad,id_usuario,accion,fecha_actividad,modulo
0,ACT9000,US518,actualizar pedido,2024-05-20 07:00:00,seguridad
1,ACT9001,US512,cambio clave,2024-11-02 12:00:00,reportes
2,ACT9002,US450,logout,2024-08-29 19:00:00,usuarios
3,ACT9003,US467,consulta,2024-04-16 14:00:00,seguridad
4,ACT9004,US420,consulta,2024-08-08 05:00:00,inventario


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_actividad     246 non-null    object
 1   id_usuario       238 non-null    object
 2   accion           246 non-null    object
 3   fecha_actividad  240 non-null    object
 4   modulo           246 non-null    object
dtypes: object(5)
memory usage: 9.7+ KB


In [3]:
df.describe(include="all")

,id_actividad,id_usuario,accion,fecha_actividad,modulo
count,246,238,246,240,246
unique,240,104,15,229,5
top,ACT9044,US440,logout,2025/99/10,inventario
freq,2,5,41,2,60


**LIMPIEZA DE DATOS**

In [4]:
# Eliminar espacios en strings
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Convertir fecha a datetime
df["fecha_actividad"] = pd.to_datetime(df["fecha_actividad"], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_actividad     246 non-null    object        
 1   id_usuario       238 non-null    object        
 2   accion           246 non-null    object        
 3   fecha_actividad  230 non-null    datetime64[ns]
 4   modulo           246 non-null    object        
dtypes: datetime64[ns](1), object(4)
memory usage: 9.7+ KB


In [5]:
#VALIDACIÓN
condicion = (
    df["id_actividad"].notnull() &
    df["id_usuario"].notnull() &
    df["accion"].notnull() &
    df["fecha_actividad"].notnull() &
    df["modulo"].notnull()
)

**CREACIÓN DE CURATES Y REJECS**

In [6]:
df_curated = df[condicion].copy()
df_rejects = df[~condicion].copy()

print("Curated:", df_curated.shape)
print("Rejects:", df_rejects.shape)

Curated: (223, 5)
Rejects: (23, 5)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [7]:
def motivo(row):
    if pd.isnull(row["id_usuario"]):
        return "id_usuario nulo"
    elif pd.isnull(row["fecha_actividad"]):
        return "fecha inválida o nula"
    else:
        return "otros"

df_rejects["motivo_rechazo"] = df_rejects.apply(motivo, axis=1)

df_rejects.head()

,id_actividad,id_usuario,accion,fecha_actividad,modulo,motivo_rechazo
14,ACT9014,NaN,login,2024-03-11 19:00:00,inventario,id_usuario nulo
15,ACT9015,US447,crear pedido,NaT,usuarios,fecha inválida o nula
29,ACT9029,US491,crear pedido,NaT,usuarios,fecha inválida o nula
30,ACT9030,NaN,logout,2024-03-22 01:00:00,usuarios,id_usuario nulo
34,ACT9034,NaN,anular pedido,2024-04-24 15:00:00,usuarios,id_usuario nulo


**EXPORTAR ARCHIVOS**

In [8]:
df_curated.to_csv("actividad_curated.csv", index=False)
df_rejects.to_csv("actividad_rejects.csv", index=False)